# Using NABOPB

A hands-on walkthrough of the `NABOPB` function: how to install the repo, how to call it, how to
read what it returns, and how to use the result. No theory. If you want to know *why* it works,
see `tutorial_nabopb.ipynb`.

What you get out: for a function $f$ of $d$ variables, the $s$ most important coefficients of
its expansion in a Fourier or Chebyshev basis, together with which basis functions they belong
to, found from point evaluations of $f$ alone.

## 0. Install

In a terminal:

```bash
git clone https://github.com/fabiantaubert/nabopb.git
cd nabopb
python3 -m venv .venv
source .venv/bin/activate            # Windows: work inside WSL, see the note in step 4
pip install -r requirements.txt ipykernel ipywidgets
```

Then open this notebook and select the `.venv` interpreter as its kernel (in VS Code: kernel
picker, top right, *Select Another Kernel*, *Python Environments*, `.venv`).

PS-

- `pykeops` prints a few warnings about CUDA and OpenMP when first imported. They are expected
  on a machine without a GPU and do not affect anything here.

In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np
from collections import Counter

REPO = Path.cwd() if (Path.cwd() / "NABOPB.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)      # one upstream file has LaTeX in plain strings
    from NABOPB import NABOPB

v = tuple(int(p) for p in np.__version__.split(".")[:2])
print(f"numpy {np.__version__}", "ok" if v < (2, 4) else "TOO NEW: run  pip install 'numpy<2.4'")
print(f"repo   {REPO}")

## 1. Provide a function

`NABOPB` needs one thing from you: a callable that takes an array of points and returns one
value per point.

> `f(x)` receives `x` of shape `(num_points, d)` and returns an array of shape `(num_points,)`.

The domain depends on the basis you will pick in step 4: $[0,1]^d$ for Fourier, $[-1,1]^d$ for
Chebyshev. Your function can be anything, including a numerical solver; the PDE examples in
`pde_applications/` do exactly that.

We use a test function that ships with the repo, `bspline_test_10d`: ten variables on
$[0,1]^{10}$, and it comes with a companion that returns its exact coefficients, so we can check
the answer.

In [2]:
from bspline_test_10d import bspline_test_10d, bspline_test_10d_fouriercoeff

d = 10
x_probe = np.random.rand(3, d)                    # three random points in [0,1]^10
print("f(x) at three points:", np.round(bspline_test_10d(x_probe), 4))

_, norm_sq = bspline_test_10d_fouriercoeff(np.zeros((1, d), dtype=int))
NORM_SQ = float(np.ravel(norm_sq)[0])
print(f"squared L2 norm of f (for accuracy checks later): {NORM_SQ:.4f}")

f(x) at three points: [0.9465 2.2697 0.9394]
squared L2 norm of f (for accuracy checks later): 3.8605


## 2. Call `NABOPB`

```python
result = NABOPB(fct_handle, d, basis_flag, gamma, diminc, sparsity_s)
```

`fct_handle` is your function from step 1 and `d` is its number of variables. `basis_flag` is a
single string that picks both the basis and the reconstruction method (step 4); we start with
`"Fourier_r1l"`. `diminc` selects the dimension-incremental strategy, and `{"type": "default"}`
is the one to use. `sparsity_s` is how many coefficients you want back; here `25`.

`gamma` is a dict describing the search space:

| key | meaning |
|---|---|
| `"type"` | `"full"` for a box, `"sym_hc"` for a hyperbolic cross |
| `"N"` | box half-width: frequencies from $-N$ to $N$ in each variable |
| `"sgn"` | `"default"` allows negative frequencies (Fourier); `"non-negative"` for Chebyshev |
| `"superpos"` | optional: keep only coefficients involving at most this many variables at once |

**Parameters used here are reduced so cells run in seconds** (the method comparison in step 4
takes about a minute). `demo.py` and the paper use
`sparsity_s = 100` and `N = 16` for this function. Be aware that the lattice methods allocate
memory proportional to the *square* of the candidate set size, so `s = 100` at `d = 10` can
exceed a laptop's RAM. Start small and increase `s`.

In [3]:
gamma = {"type": "full", "N": 8, "sgn": "default", "superpos": 4}
diminc = {"type": "default"}
s = 25

np.random.seed(0)
t0 = time.time()
result = NABOPB(bspline_test_10d, d, "Fourier_r1l", gamma, diminc, sparsity_s=s)
print(f"\ndone in {time.time() - t0:.1f}s")
print("result keys:", list(result))

Algorithm Start: d = 10, basis and method = Fourier_r1l, diminc. approach = default
Algorithm Start: Gamma type = full
Algorithm Start: sparsity s = 25, sparsity s_local = 25, delta = 1.000e-12, det. iterations r = 5
Step 1:
dim [1] done, card(I)=17
Step 2:
dim [2] done, card(I)=17
dim-inc of [[1]] and [[2]] done, card(I)=35
dim [3] done, card(I)=17
dim-inc of [[1 2]] and [[3]] done, card(I)=29
dim [4] done, card(I)=17
dim-inc of [[1 2 3]] and [[4]] done, card(I)=35
dim [5] done, card(I)=17
dim-inc of [[1 2 3 4]] and [[5]] done, card(I)=47
dim [6] done, card(I)=17


dim-inc of [[1 2 3 4 5]] and [[6]] done, card(I)=51
dim [7] done, card(I)=17
dim-inc of [[1 2 3 4 5 6]] and [[7]] done, card(I)=63
dim [8] done, card(I)=17


dim-inc of [[1 2 3 4 5 6 7]] and [[8]] done, card(I)=61
dim [9] done, card(I)=17
dim-inc of [[1 2 3 4 5 6 7 8]] and [[9]] done, card(I)=43
dim [10] done, card(I)=17


Algorithm finished with 25 detected indices.



done in 0.8s
result keys: ['index', 'val', 'sample_sizes', 'cand_sizes', 'run_times']


The lines printed during the run are progress: one `dim j done` per variable, then one
`dim-inc` line per merge step, each reporting how many candidate coefficients survived.

## 3. Read the result

`result` is a dict:

| key | type | meaning |
|---|---|---|
| `index` | int array `(s, d)` | the detected frequencies, one row per coefficient |
| `val` | complex array `(s,)` | the coefficient for each row of `index` |
| `sample_sizes` | array `(d,)` | function evaluations spent at each dimension-increment step |
| `cand_sizes` | array `(d,)` | candidate coefficients examined at each step |
| `run_times` | array `(4,)` | seconds for node construction, sampling, reconstruction, total |

Since we know this function's true coefficients, we can grade the answer.

In [4]:
index, val = result["index"], result["val"]

true_val, _ = bspline_test_10d_fouriercoeff(index)          # exact coefficients at the detected indices
err = np.abs(np.real(val) - true_val)
captured = float(np.sum(np.abs(val) ** 2)) / NORM_SQ

print(f"detected {index.shape[0]} coefficients using {int(np.sum(result['sample_sizes'])):,} function evaluations")
print(f"largest coefficient error : {err.max():.2e}")
print(f"share of f's energy found : {100 * captured:.1f}%   (sum |val|^2 / ||f||^2)")
print(f"time: sampling {result['run_times'][1]:.3f}s, reconstruction {result['run_times'][2]:.3f}s, total {result['run_times'][3]:.2f}s")

print(f"\nthe ten largest terms   (frequency vector k, then coefficient, then the true value)")
for i in np.argsort(-np.abs(val))[:10]:
    print(f"   {str(index[i]):<32} {np.real(val[i]):>10.5f}   true {true_val[i]:>10.5f}")

detected 25 coefficients using 168,461 function evaluations
largest coefficient error : 4.32e-02
share of f's energy found : 64.8%   (sum |val|^2 / ||f||^2)
time: sampling 0.052s, reconstruction 0.013s, total 0.84s

the ten largest terms   (frequency vector k, then coefficient, then the true value)
   [0 0 0 0 0 0 0 0 0 0]               1.19672   true    1.19671
   [-1  0  0  0  0  0  0  0  0  0]    -0.26324   true   -0.26324
   [1 0 0 0 0 0 0 0 0 0]              -0.26324   true   -0.26324
   [0 0 1 0 0 0 0 0 0 0]              -0.26323   true   -0.26324
   [ 0  0 -1  0  0  0  0  0  0  0]    -0.26323   true   -0.26324
   [0 0 0 0 0 0 0 1 0 0]              -0.26322   true   -0.26324
   [ 0  0  0  0  0  0  0 -1  0  0]    -0.26322   true   -0.26324
   [0 0 0 0 0 0 0 0 0 1]              -0.22195   true   -0.17870
   [ 0  0  0  0  0  0  0  0  0 -1]    -0.22195   true   -0.17870
   [0 0 0 1 0 0 0 0 0 0]              -0.20863   true   -0.20868


Look at which positions are nonzero in the frequency vectors: never more than two per row, and
the positions fall into three disjoint groups of variables ({1, 3, 8}, {2, 5, 6, 10}, {4, 7, 9},
counting from 1). That is genuine structure in this function, and `NABOPB` found it from
samples alone; `superpos = 4` only capped how many nonzeros were *allowed*.

## 4. Choose a method

`basis_flag` picks two things at once: the basis, and how coefficients are recovered from
samples. For a Fourier basis on $[0,1]^d$ the options are:

| flag | sampling | when to use |
| --- | --- | --- |
| `Fourier_r1l` | one rank-1 lattice | very fast FFT reconstruction; simple and stable, but requires many samples |
| `Fourier_ssr1l` | subsampled rank-1 lattice | fewer samples than `Fourier_r1l`; iterative least squares accelerated by lattice FFTs |
| `Fourier_mr1l` | several small rank-1 lattices | good lattice compromise; relatively few samples and fast structured reconstruction |
| `Fourier_rand` | random points | simple and flexible baseline; dense least squares and comparatively many samples |
| `Fourier_OMP` | random points | very sample-efficient; simple greedy sparse recovery, but computationally slower |
| `Fourier_COSAMP` | random points | very sample-efficient; more robust greedy recovery, with higher per-iteration cost |
| `Fourier_SRLasso` | random points | very sample-efficient; suited to compressible or noisy problems, but typically slower |

The Chebyshev family on $[-1,1]^d$ mirrors this with `Cheby_` in place of `Fourier_`:
`Cheby_rand`, `Cheby_mr1l`, `Cheby_mr1l_subsampling`, `Cheby_OMP`, `Cheby_COSAMP` and
`Cheby_SRLasso`.

**Note.** The last three methods compile small C++ kernels through KeOps the first time they are
used. That takes a minute or two, once, and needs a working C++ compiler (`g++` or `clang++`). On a
first run that time is included in the row of `Fourier_OMP` below.
If the compiler is missing or broken, for example on a Mac whose Xcode is older than its macOS,
they stop with `ModuleNotFoundError: No module named 'pykeops_cpp_...'`, while the four
methods above them keep working. On native Windows KeOps cannot be imported at all, and since
this repository imports it at start-up, nothing here runs there; use WSL.

Same function, same `gamma`, all seven methods. Six are run here. `Fourier_SRLasso` needs
about twelve minutes at these settings, so its row is a recorded measurement.

In [ ]:
import contextlib, io

np.random.seed(123)
X_cmp = np.random.rand(3000, d)                      # new points, to measure the error at
f_cmp = bspline_test_10d(X_cmp)

def run(flag):
    np.random.seed(0)
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):  # hide the progress lines
        r = NABOPB(bspline_test_10d, d, flag, gamma, diminc, sparsity_s=s)
    tv, _ = bspline_test_10d_fouriercoeff(r["index"])
    approx = np.real(np.exp(2j * np.pi * (X_cmp @ r["index"].T)) @ r["val"])  # explained in step 5
    return (int(np.sum(r["sample_sizes"])),                                 # evaluations
            100 * float(np.sum(np.abs(r["val"]) ** 2)) / NORM_SQ,           # energy found, percent
            np.max(np.abs(np.real(r["val"]) - tv)),                         # max coefficient error
            np.linalg.norm(approx - f_cmp) / np.linalg.norm(f_cmp),         # error at new points
            time.time() - t0)

row = "{:<17}{:>12,}{:>13.1f}%{:>14.1e}{:>16.2f}{:>9.1f}s"
print(f"{'flag':<17}{'evaluations':>12}{'energy found':>14}{'max coef err':>14}{'error, new pts':>16}{'time':>10}")
print("-" * 83)
for flag in ["Fourier_r1l", "Fourier_ssr1l", "Fourier_mr1l", "Fourier_rand",
             "Fourier_OMP", "Fourier_COSAMP"]:
    try:
        print(row.format(flag, *run(flag)))
    except ModuleNotFoundError as missing:               # KeOps could not compile, see the note above
        print(f"{flag:<17}   not available on this machine ({missing.name})")

# Recorded once on an Apple M2/ Laptop (KeOps on CPU, single-threaded), not run here.
# To run it yourself: print(row.format("Fourier_SRLasso", *run("Fourier_SRLasso")))
print(row.format("Fourier_SRLasso", 16500, 67.5, 0.21, 0.71, 745.1) + "   (recorded at s = 25, N = 8, superpos = 4)")

flag              evaluations  energy found  max coef err  error, new pts      time
-----------------------------------------------------------------------------------


Fourier_r1l           168,461         64.8%       4.3e-02            0.60      0.8s


Fourier_ssr1l          76,956         67.5%       3.3e-02            0.60      0.9s


Fourier_mr1l           57,023         61.0%       4.5e-02            0.62      0.7s


Fourier_rand          160,034         58.9%       3.2e-02            0.62     12.8s


Fourier_OMP            16,500         97.6%       3.5e-01            0.89      2.1s


Fourier_COSAMP         16,500        105.2%       4.1e-01            1.07     18.6s
Fourier_SRLasso        16,500         67.5%       2.1e-01            0.71    745.1s   (recorded at s = 25, N = 8, superpos = 4)


Read the table in two halves.

**The first four methods** end up equally accurate, with an error of about 0.6 at new points,
and differ in what they cost. In this run `Fourier_mr1l` needs a third of the evaluations
`Fourier_r1l` does; the counts depend on the random seed, and a factor of 1.5 to 2 is more
typical. When `f` is cheap that hardly matters. When `f` is a PDE solve at a second per call, a
factor of two or three in evaluations is a factor of two or three in waiting time, and
`Fourier_mr1l` or `Fourier_ssr1l` is the right choice.

**The three compressed-sensing methods** use 16,500 evaluations, between three and ten times
fewer than the others, and here they pay for it: coefficient errors of 0.2 to 0.4 against 0.03
to 0.05, and a larger error at new points. For `Fourier_COSAMP` that error is above 1, which is
worse than approximating `f` by zero.

The reason is how these methods size their sample: `10 * s` points per step, whatever the size
of the candidate set. With `s = 25` that is 250 points to tell apart several hundred to a
thousand candidates, on a function that is far from 25-sparse: more than 500 of its coefficients
are larger than 0.01. They are built for large `s`: the heat, Burgers and diffusion examples in
`pde_applications/` use `s = 1000`. There their sample count still grows only in proportion to
`s`, while a rank-1 lattice grows with the square of the candidate set and runs out of memory,
as noted in step 2. At the small scale of this tutorial, the lattice methods are the better tool.

**"Energy found" can only be trusted when the coefficients are accurate.** No 25-term
approximation of this function can capture more than about 64 percent of its energy. The first
four rows lie within a few points of that ceiling, on either side of it, because even a
coefficient error of 0.03 moves the figure by a few points. The 98 and 105 percent further down
are coefficient error, not information, and the 67.5 percent of `Fourier_SRLasso` comes with a
coefficient error of 0.2, so it deserves no more trust. Check the coefficient error, or the
error at new points, before relying on it.

## 5. Use the approximation

`index` and `val` define a function you can evaluate anywhere:

$$\tilde f(\mathbf{x}) \;=\; \sum_{k} \text{val}_k \, e^{2\pi i \langle \mathbf{k}, \mathbf{x}\rangle}$$

For a real-valued `f`, take the real part. One line of numpy:

In [6]:
def evaluate_fourier(index, val, X):
    """Evaluate the approximation at points X of shape (num_points, d) on [0,1]^d."""
    return np.real(np.exp(2j * np.pi * (X @ index.T)) @ val)

np.random.seed(123)
X_new = np.random.rand(3000, d)
f_true = bspline_test_10d(X_new)
f_approx = evaluate_fourier(index, val, X_new)

rel_err = np.linalg.norm(f_approx - f_true) / np.linalg.norm(f_true)
print(f"relative L2 error at 3000 new points : {rel_err:.3f}")
print(f"predicted from the energy found      : {np.sqrt(1 - captured):.3f}   (sqrt(1 - share))")

print("\nfirst five points:")
print(f"   {'true':>9}{'approx':>9}")
for a, b in zip(f_true[:5], f_approx[:5]):
    print(f"   {a:>9.4f}{b:>9.4f}")

top = f_true >= np.quantile(f_true, 0.9)                     # the 10% of points where f is largest
resid = f_approx - f_true
print(f"\nmax over these points : true {f_true.max():.2f}, approximation {f_approx.max():.2f}")
print(f"share of squared error at those top-10% points : {100 * np.sum(resid[top]**2) / np.sum(resid**2):.0f}%")

relative L2 error at 3000 new points : 0.602
predicted from the energy found      : 0.593   (sqrt(1 - share))

first five points:
        true   approx
      2.1370   2.0850
      2.0820   2.1274
      1.8657   2.6826
      5.4074   1.9116
      7.7019   2.9120

max over these points : true 11.07, approximation 4.89
share of squared error at those top-10% points : 63%


Individual points can be far off, and the last two lines say where: about two thirds of the squared
error sits at the ten percent of points where `f` is largest, and there the approximation is too
low, its maximum reaching roughly half the true one. A 25-term Fourier sum is a smooth function;
the sharp peaks of a tensor B-spline live in high frequencies that were not among the 25. Away
from the peaks it tracks `f`. That is what capturing 65 percent of the energy looks like.

The *predicted* line is the one to remember. You will not normally have the true function to
compare against, but `sum(abs(val)**2)` is always available, and the relative error of the
approximation is close to $\sqrt{1 - \text{share of energy found}}$, because the error is
dominated by the coefficients you did not ask for rather than by inaccuracy in the ones you did.
That holds for the first four methods of step 4, not for the compressed-sensing ones, where
the coefficients themselves are inaccurate.
So **`s` is your accuracy knob**: `s = 25` captures about 65 percent of this function's energy;
the paper's `s = 100` captures considerably more, at the cost of samples and memory noted in
step 2.

## 6. The same call with a Chebyshev basis

For functions on $[-1,1]^d$ use the Chebyshev family. Three things change in the call and
nothing else: the flag (`Cheby_mr1l`), `"sgn": "non-negative"` because Chebyshev indices have
no sign, and the domain your function expects.

The repo's second test function, `bsplinet_test_9d`, is a sum of two tensor B-splines on
$[-1,1]^9$, and `bsplinet_test_9d_chat` returns its exact coefficients. `demo.py` runs it with
`s = 100`, `N = 16` and three flags; we use `s = 25`, `N = 8` and `Cheby_mr1l`, which needs
about one fifteenth of the function evaluations `Cheby_rand` does on this problem.

In [7]:
from bsplinet_test_9d import bsplinet_test_9d, bsplinet_test_9d_chat

d9 = 9
gamma9 = {"type": "full", "N": 8, "sgn": "non-negative", "superpos": 5}

np.random.seed(0)
t0 = time.time()
res9 = NABOPB(bsplinet_test_9d, d9, "Cheby_mr1l", gamma9, diminc, sparsity_s=25)
index9, val9 = res9["index"], res9["val"]
print(f"\ndone in {time.time() - t0:.1f}s, {int(np.sum(res9['sample_sizes'])):,} function evaluations")

Algorithm Start: d = 9, basis and method = Cheby_mr1l, diminc. approach = default
Algorithm Start: Gamma type = full
Algorithm Start: sparsity s = 25, sparsity s_local = 25, delta = 1.000e-12, det. iterations r = 5
Step 1:
dim [1] done, card(I)=9
Step 2:
dim [2] done, card(I)=9
dim-inc of [[1]] and [[2]] done, card(I)=28
dim [3] done, card(I)=9


dim-inc of [[1 2]] and [[3]] done, card(I)=27
dim [4] done, card(I)=9


dim-inc of [[1 2 3]] and [[4]] done, card(I)=25
dim [5] done, card(I)=9


dim-inc of [[1 2 3 4]] and [[5]] done, card(I)=25
dim [6] done, card(I)=9


dim-inc of [[1 2 3 4 5]] and [[6]] done, card(I)=28
dim [7] done, card(I)=9


dim-inc of [[1 2 3 4 5 6]] and [[7]] done, card(I)=27
dim [8] done, card(I)=9


dim-inc of [[1 2 3 4 5 6 7]] and [[8]] done, card(I)=25
dim [9] done, card(I)=9
Algorithm finished with 25 detected indices.



done in 1.8s, 25,044 function evaluations


One convention to know when grading Chebyshev results. `NABOPB` returns coefficients for the
*orthonormal* basis, whose functions carry a factor $\sqrt{2}$ for every nonzero index entry.
`bsplinet_test_9d_chat` returns coefficients for the plain products of Chebyshev polynomials,
without those factors. To compare, multiply `val` by $\sqrt{2}$ per nonzero entry, exactly as
`demo.py` does.

In [8]:
scale9 = np.sqrt(2.0) ** np.count_nonzero(index9, axis=1)
true9, _ = bsplinet_test_9d_chat(index9)
print(f"largest coefficient error: {np.max(np.abs(np.real(val9) * scale9 - true9)):.1e}")

print("\nthe six largest terms   (index k, coefficient, true value)")
for i in np.argsort(-np.abs(val9))[:6]:
    print(f"   {str(index9[i]):<30} {np.real(val9[i]) * scale9[i]:>10.6f}   true {true9[i]:>10.6f}")

largest coefficient error: 2.1e-03

the six largest terms   (index k, coefficient, true value)
   [0 0 0 0 0 0 0 0 0]              0.011543   true   0.011555
   [1 0 0 0 0 0 0 0 0]             -0.012219   true  -0.012120
   [0 0 0 1 0 0 0 0 0]             -0.012117   true  -0.012120
   [0 0 0 0 0 0 1 0 0]             -0.012066   true  -0.012120
   [0 0 1 0 0 0 0 0 0]             -0.011945   true  -0.012120
   [1 0 1 0 0 0 0 0 0]              0.012780   true   0.012764


/Users/gauravkumar/Downloads/naopbv2/nabopb/bsplinet_test_9d.py:95: RuntimeWarning: divide by zero encountered in divide
  chat = (
/Users/gauravkumar/Downloads/naopbv2/nabopb/bsplinet_test_9d.py:95: RuntimeWarning: invalid value encountered in divide
  chat = (
/Users/gauravkumar/Downloads/naopbv2/nabopb/bsplinet_test_9d.py:182: RuntimeWarning: invalid value encountered in divide
  chat = (


Evaluating the approximation needs the same factor. The basis function for index $k$ is
$\sqrt{2}^{\,\#\{j:\,k_j \neq 0\}} \prod_j T_{k_j}(x_j)$ with $T_n(x) = \cos(n \arccos x)$.

Chebyshev methods weight the domain toward the endpoints, so test points, and any norm you
estimate from them, should be drawn the same way: $x = \sin(\pi(u - \tfrac12))$ for uniform $u$.

In [9]:
def evaluate_chebyshev(index, val, X):
    """Evaluate the approximation at points X of shape (num_points, d) on [-1,1]^d."""
    T = np.cos(index[None, :, :] * np.arccos(X)[:, None, :])        # T_k(x_j) for each point, index, variable
    scale = np.sqrt(2.0) ** np.count_nonzero(index, axis=1)
    return np.real(np.prod(T, axis=2) @ (val * scale))

cheb_points = lambda n, d: np.sin(np.pi * (np.random.rand(n, d) - 0.5))

np.random.seed(123)
X9 = cheb_points(20000, d9)
f9_true, f9_approx = bsplinet_test_9d(X9), evaluate_chebyshev(index9, val9, X9)
rel9 = np.linalg.norm(f9_approx - f9_true) / np.linalg.norm(f9_true)

norm9_sq = np.mean(bsplinet_test_9d(cheb_points(400000, d9)) ** 2)   # ||f||^2 under the Chebyshev measure
share9 = float(np.sum(np.abs(val9) ** 2)) / norm9_sq

print(f"relative L2 error at 20000 new points : {rel9:.3f}")
print(f"predicted from the energy found       : {np.sqrt(max(0.0, 1 - share9)):.3f}   (share = {100 * share9:.1f}%)")

relative L2 error at 20000 new points : 0.121
predicted from the energy found       : 0.149   (share = 97.8%)


Twenty-five coefficients capture about 98 percent of this function, against 65 percent for the
10-d one with the same `s`. Both are B-splines; this one is smoother (fourth order in five of its
variables), and smoother functions have faster-decaying coefficients, so the same budget goes
much further. How far a given `s` gets you depends on the function, and the predicted-error
line tells you without knowing the answer. It is a rougher guide here than in step 5: when
nearly everything is captured, $\sqrt{1 - \text{share}}$ is very sensitive, and a one percent
error in the norm estimate or in the coefficients moves it by about 0.03, which is the gap you
see between predicted and measured.

A note on the norm estimate above. `bsplinet_test_9d_chat` also returns a squared norm for this
function, but that value does not match the function as implemented (it is about 3.5 times too
large), and using it would make the predicted error read 0.85 instead of about 0.15. Estimating
$\|f\|^2$ from Chebyshev-distributed points sidesteps that, and in practice you rarely have a
reference norm anyway.

## That is the whole workflow

For your own function: write `f(x)` for `(num_points, d)` input, pick Fourier on $[0,1]^d$ or
Chebyshev on $[-1,1]^d$, start with `Fourier_r1l` or `Cheby_mr1l` and a small `s`, read
`sum(abs(val)**2)` against an estimate of $\|f\|^2$ to see how much you captured, and raise `s`
until that is enough. `demo.py` has these two examples at paper scale, and `pde_applications/`
shows the same call with PDE solvers as `f`.